Layer: Transformation → Silver
Source: Bronze (Delta Tables)
Target: Cleaned Silver Table



 1 Read Bronze Table


In [0]:
df_bronze_invoices = spark.read.table("workspace.bronze.erp_invoices") 

In [0]:
# Preview data
display(df_bronze_invoices)

# Check schema
df_bronze_invoices.printSchema()

In [0]:
df_bronze_invoices.columns

 NOTE:
 Column naming follows best practices:
 - snake_case format
 - lowercase letters only
 - descriptive and explicit names
   e.g., order_id, invoice_id, amount_paid


 2 Data Type Casting



In [0]:
import pyspark.sql.functions as F

df_typed = df_bronze_invoices \
    .withColumn("invoice_id", F.col("invoice_id").cast("string")) \
    .withColumn("order_id", F.col("order_id").cast("string")) \
    .withColumn("invoice_date", F.to_date(F.col("invoice_date"), "dd/MM/yyyy")) \
    .withColumn("amount_paid", F.col("amount_paid").cast("double")) \
    .withColumn("payment_method", F.col("payment_method").cast("string")) \
    .withColumn("payment_status", F.col("payment_status").cast("string")) \
    .withColumn("ingestion_timestamp", F.col("ingestion_timestamp").cast("timestamp")) \
    .withColumn("source_file", F.col("source_file").cast("string"))

df_typed.printSchema()



 3 Handle Null Values



 DATA QUALITY CONSTRAINTS (ERP - Invoices)

#
 Business rules applied in Silver layer:
#
- invoice_id must be unique
- order_id cannot be NULL
- amount_paid >= 0
- invoice_date must be a valid date
- payment_status must be one of: (Pending, Paid, Partial)

In [0]:
df_typed.summary("count").show()

In [0]:
df_clean = df_typed \
    .filter(F.col("invoice_id").isNotNull()) \
    .filter(F.col("order_id").isNotNull()) \
    .filter(F.col("invoice_date").isNotNull()) \
    .withColumn(
        "amount_paid",
        F.when(F.col("amount_paid").isNull(), 0).otherwise(F.col("amount_paid"))
    ) \
    .withColumn(
        "payment_method",
        F.when(F.col("payment_method").isNull(), "unknown").otherwise(F.col("payment_method"))
    ) \
    .withColumn(
        "payment_status",
        F.when(F.col("payment_status").isNull(), "Pending").otherwise(F.col("payment_status"))
    )


 4 Data Quality Rules Enforcement


In [0]:
df_valid = df_clean \
    .filter(F.col("amount_paid") >= 0) \
    .filter(F.col("invoice_date").isNotNull()) \
    .filter(F.col("payment_status").isin("Pending", "Paid", "Partial"))


 5 Deduplication 


In [0]:
from pyspark.sql.window import Window

window_spec = Window.partitionBy("invoice_id").orderBy(F.col("ingestion_timestamp").desc())

df_dedup = df_valid \
    .withColumn("row_num", F.row_number().over(window_spec)) \
    .filter(F.col("row_num") == 1) \
    .drop("row_num")

In [0]:
df_dedup.count()
df_dedup.display()


 6 Write to Silver Table


In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS silver.invoices (
    invoice_id STRING,
    order_id STRING,
    invoice_date DATE,
    amount_paid DOUBLE,
    payment_method STRING,
    payment_status STRING,
    ingestion_timestamp TIMESTAMP,
    source_file STRING
)
USING DELTA
""")

In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "silver.invoices")

(
    delta_table.alias("target")
    .merge(
        df_dedup.alias("source"),
        "target.invoice_id = source.invoice_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

In [0]:
spark.read.table("silver.invoices").count()

In [0]:
display(table("silver.invoices"))
